# Train and Compare LoRA vs DoRA on Google Colab

This notebook helps you to:
1. Connect Google Drive to retrieve Nghệ Tĩnh dialect speech data from the `NA_HT_Whisper` directory.
2. **Train a traditional LoRA model** (saved to `whisper-lora-nghe-tinh folder inside your Google Drive`).
3. **Train a DoRA (Weight-Decomposed LoRA) model** (saved to `whisper-dora-nghe-tinh folder inside your Google Drive`).
4. **Directly compare side-by-side results** of both models on a Gradio interface to evaluate which model transcribes more accurately.

**Note:** Please select **T4 GPU** in the Runtime settings before running (Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU).

### Step 1: Connect Google Drive and Set Working Directory

Run the cell below to mount your Google Drive. We will change the working directory directly to the `NA_HT_Whisper` folder on your Drive to automatically read the data and save the training results.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to your project directory on Google Drive
project_dir = "/content/drive/MyDrive/NA_HT_Whisper"

# Automatically create the project directory on Drive if it doesn't exist to ensure the model is saved to Drive
os.makedirs(project_dir, exist_ok=True)
%cd {project_dir}
print(f"Working directory changed to: {project_dir}")

Mounted at /content/drive
/content/drive/MyDrive/NA_HT_Whisper
Working directory changed to: /content/drive/MyDrive/NA_HT_Whisper


### Step 2: Install Required Libraries

In [2]:
!pip install transformers datasets accelerate peft evaluate jiwer librosa soundfile gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.2 MB/s eta 0:00:00


In [3]:
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### Step 3: Data Loading and Preprocessing Function (Shared between both training runs)

In [ ]:
import os
import json
import torch
import librosa
from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# --- GENERAL CONFIGURATION ---
MODEL_NAME = "vinai/phowhisper-base"
LABELS_JSON_PATH = "metadata/labels.json"
TEXT_COLUMN = "dialect_text"  # Switched to learn original Nghệ Tĩnh dialect (dialect_text) instead of standard_text
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 1e-4

def load_custom_dataset(labels_path):
    with open(labels_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    valid_data = []
    for item in data:
        audio_path = item.get("audio_path")
        if audio_path and os.path.exists(audio_path):
            valid_data.append({
                "audio": audio_path,
                "text": item.get(TEXT_COLUMN, "")
            })

    print(f"Loaded {len(valid_data)} valid samples from {labels_path}")
    dataset = Dataset.from_list(valid_data)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    return dataset

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# Initialize processor and prepare data
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="vi", task="transcribe")
raw_dataset = load_custom_dataset(LABELS_JSON_PATH)
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("Preprocessing audio data...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=raw_dataset.column_names, num_proc=1)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=raw_dataset.column_names, num_proc=1)

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

Loaded 359 valid samples from metadata/labels.json
Preprocessing audio data...


Map:   0%|          | 0/323 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

### Step 4.1: Train Traditional LoRA

Training results and checkpoints for this run will be saved at `whisper-lora-nghe-tinh folder inside your Google Drive`.

In [ ]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
EPOCHS = 20

In [ ]:
import os
OUTPUT_LORA_DIR = os.path.join(project_dir, "whisper-lora-nghe-tinh")

# 1. Load Base Model
print("Loading base model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, device_map="auto")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 2. Configure LoRA (use_dora = False)
print("Applying standard LoRA configuration...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    use_dora=False  # Use standard LoRA
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_LORA_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=-1,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=True if torch.cuda.is_available() else False,
    dataloader_num_workers=0
)

# Check for existing checkpoints to automatically resume training
resume_from_checkpoint = None
if os.path.exists(OUTPUT_LORA_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_LORA_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        latest_checkpoint = os.path.join(OUTPUT_LORA_DIR, checkpoints[-1])
        resume_from_checkpoint = latest_checkpoint
        print(f"Found checkpoint: {latest_checkpoint}. Automatic resume enabled.")
    else:
        print("No checkpoint found. Training from scratch.")
else:
    print("Output directory does not exist. Training from scratch.")

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting traditional LoRA training...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)

# Save final LoRA model
trainer.model.save_pretrained(OUTPUT_LORA_DIR)
processor.save_pretrained(OUTPUT_LORA_DIR)
print("Finished traditional LoRA training!")

Loading base model...


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Applying standard LoRA configuration...
trainable params: 294,912 || all params: 99,443,712 || trainable%: 0.2966
Found checkpoint: /content/drive/MyDrive/NA_HT_Whisper/whisper-lora-nghe-tinh/checkpoint-410. Automatic resume enabled.
Starting traditional LoRA training...


Epoch,Training Loss,Validation Loss
11,1.731063,1.280996
12,1.456558,1.254855
13,1.419484,1.236416
14,1.472990,1.224735
15,1.346193,1.211758
16,1.479727,1.201317
17,1.324630,1.194990
18,1.483704,1.188409
19,1.365174,1.186442
20,1.351688,1.185825


Finished traditional LoRA training!


### Step 4.2: Train DoRA (Weight-Decomposed LoRA)

Training results and checkpoints for this run will be saved at `whisper-dora-nghe-tinh folder inside your Google Drive`.

In [ ]:
import os
OUTPUT_DORA_DIR = os.path.join(project_dir, "whisper-dora-nghe-tinh")

# 1. Load new Base Model
print("Loading base model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, device_map="auto")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 2. Configure DoRA (use_dora = True)
print("Applying DoRA (Weight-Decomposed LoRA) configuration...")
dora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    use_dora=True  # Enable DoRA
)
model = get_peft_model(model, dora_config)
model.print_trainable_parameters()

# 3. Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DORA_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=-1,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=True if torch.cuda.is_available() else False,
    dataloader_num_workers=0
)

# Check for existing checkpoints to automatically resume training
resume_from_checkpoint = None
if os.path.exists(OUTPUT_DORA_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DORA_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        latest_checkpoint = os.path.join(OUTPUT_DORA_DIR, checkpoints[-1])
        resume_from_checkpoint = latest_checkpoint
        print(f"Found checkpoint: {latest_checkpoint}. Automatic resume enabled.")
    else:
        print("No checkpoint found. Training from scratch.")
else:
    print("Output directory does not exist. Training from scratch.")

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting DoRA training...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)

# Save final DoRA model
trainer.model.save_pretrained(OUTPUT_DORA_DIR)
processor.save_pretrained(OUTPUT_DORA_DIR)
print("Finished DoRA training!")

Loading base model...


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Applying DoRA (Weight-Decomposed LoRA) configuration...
trainable params: 313,344 || all params: 99,462,144 || trainable%: 0.3150
Output directory does not exist. Training from scratch.
Starting DoRA training...


Epoch,Training Loss,Validation Loss
1,3.400766,2.815948
2,2.385748,1.878841
3,1.931862,1.570007
4,1.788395,1.437444
5,1.904037,1.351860
6,1.618760,1.294506
7,1.692211,1.250086
8,1.603941,1.215854
9,1.550596,1.191729
10,1.465557,1.167918


Finished DoRA training!


### Step 5: Side-by-Side Comparison of LoRA vs DoRA on Gradio

Run the cell below to launch the web app. You can upload or record an audio file in the Nghệ Tĩnh dialect, and the app will perform inference simultaneously on both **LoRA** and **DoRA** models, displaying the transcription results side-by-side for direct comparison.

In [4]:
import torch
import librosa
import gradio as gr
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel, PeftConfig
import os

peft_lora_id = os.path.join(project_dir, "whisper-lora-nghe-tinh")
peft_dora_id = os.path.join(project_dir, "whisper-dora-nghe-tinh")

print("Loading base model and attaching both Adapters...")
config = PeftConfig.from_pretrained(peft_lora_id)
processor = WhisperProcessor.from_pretrained(config.base_model_name_or_path, language="vi", task="transcribe")

base_model = WhisperForConditionalGeneration.from_pretrained(
    config.base_model_name_or_path,
    device_map="auto"
)

# Load multiple adapters
model = PeftModel.from_pretrained(base_model, peft_lora_id, adapter_name="lora")
model.load_adapter(peft_dora_id, adapter_name="dora")
model.eval()
print("Comparison model is ready!")

def transcribe_comparison(audio_path):
    if audio_path is None:
        return "Audio file not found.", "Audio file not found."

    try:
        # Load audio file and resample to 16kHz
        audio_input, sr = librosa.load(audio_path, sr=16000)
        input_features = processor(audio_input, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

        with torch.no_grad():
            # 1. Switch to LoRA adapter and generate results
            model.set_adapter("lora")
            # Sửa lỗi: Truyền input_features dưới dạng keyword argument
            lora_ids = model.generate(input_features=input_features)
            lora_text = processor.batch_decode(lora_ids, skip_special_tokens=True)[0]

            # 2. Switch to DoRA adapter and generate results
            model.set_adapter("dora")
            # Sửa lỗi: Truyền input_features dưới dạng keyword argument
            dora_ids = model.generate(input_features=input_features)
            dora_text = processor.batch_decode(dora_ids, skip_special_tokens=True)[0]

        return lora_text, dora_text
    except Exception as e:
        return f"Error: {e}", f"Error: {e}"

# Build the side-by-side interface
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# Compare Nghệ Tĩnh Dialect Transcription: LoRA vs DoRA")
    gr.Markdown("Record directly or upload an audio file to evaluate the side-by-side transcription of both adapters.")

    with gr.Row():
        audio_input = gr.Audio(type="filepath", label="Nghệ Tĩnh Dialect Voice Input")

    with gr.Row():
        # Thêm lines=6 để ô hiển thị to hơn, dễ nhìn hơn
        lora_output = gr.Textbox(
            label="Traditional LoRA Transcription Result",
            interactive=False,
            lines=6
        )
        dora_output = gr.Textbox(
            label="DoRA (Weight-Decomposed LoRA) Transcription Result",
            interactive=False,
            lines=6
        )

    submit_btn = gr.Button("Transcribe")
    submit_btn.click(fn=transcribe_comparison, inputs=audio_input, outputs=[lora_output, dora_output])

demo.launch(share=True)


Loading base model and attaching both Adapters...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

Comparison model is ready!


/tmp/ipykernel_6328/2781883343.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme="soft") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa68fec3addd80b09a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
